# Titanic ML Pipeline: Find and Fix Data Leakage

This notebook demonstrates a common form of **data leakage** in a Titanic survival prediction pipeline, explains why it is a problem, and shows the corrected version.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer
from sklearn.model_selection import cross_val_score
import warnings
warnings.filterwarnings('ignore')

---
## Part 1: Load the Data

In [ ]:
train = pd.read_csv('train.csv')
test  = pd.read_csv('test.csv')

print('Train shape:', train.shape)
print('Test shape: ', test.shape)
print()
print('Missing values in train:')
print(train[['Age', 'Embarked', 'Fare']].isnull().sum())
print()
print('Missing values in test:')
print(test[['Age', 'Embarked', 'Fare']].isnull().sum())

---
## Part 2: BUGGY Pipeline (Contains Data Leakage)

This is the version with the bug. Read through the preprocessing carefully.

In [ ]:
# ---- BUGGY PIPELINE ----

train_buggy = train.copy()
test_buggy  = test.copy()

# Step 1: Combine train and test before computing fill statistics
# BUG: test data is included when computing the median and mode below
combined = pd.concat([train_buggy, test_buggy], axis=0, ignore_index=True)

# Step 2: Compute fill values from the COMBINED dataset  <-- LEAKAGE IS HERE
age_median_buggy      = combined['Age'].median()       # uses test Age values
embarked_mode_buggy   = combined['Embarked'].mode()[0] # uses test Embarked values
fare_median_buggy     = combined['Fare'].median()      # uses test Fare values

print(f"[BUGGY]  Age median computed on combined data:  {age_median_buggy}")
print(f"[BUGGY]  Embarked mode computed on combined data: {embarked_mode_buggy}")

# Step 3: Fill missing values using those tainted statistics
train_buggy['Age']      = train_buggy['Age'].fillna(age_median_buggy)
train_buggy['Embarked'] = train_buggy['Embarked'].fillna(embarked_mode_buggy)
test_buggy['Age']       = test_buggy['Age'].fillna(age_median_buggy)
test_buggy['Embarked']  = test_buggy['Embarked'].fillna(embarked_mode_buggy)
test_buggy['Fare']      = test_buggy['Fare'].fillna(fare_median_buggy)

# Step 4: Feature engineering
for df in [train_buggy, test_buggy]:
    df['Sex']      = (df['Sex'] == 'male').astype(int)
    df['Embarked'] = df['Embarked'].map({'S': 0, 'C': 1, 'Q': 2}).fillna(0)

features = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']

X_train_buggy = train_buggy[features]
y_train_buggy = train_buggy['Survived']
X_test_buggy  = test_buggy[features]

# Step 5: Train and evaluate
model_buggy = LogisticRegression(max_iter=200)
cv_scores_buggy = cross_val_score(model_buggy, X_train_buggy, y_train_buggy, cv=5)
print(f"\n[BUGGY] Cross-val accuracy: {cv_scores_buggy.mean():.4f} (+/- {cv_scores_buggy.std():.4f})")

---
## Part 3: Identifying the Leakage

### Where does the leakage happen?

The leakage occurs at the lines:
```python
combined = pd.concat([train_buggy, test_buggy], axis=0)
age_median_buggy    = combined['Age'].median()
embarked_mode_buggy = combined['Embarked'].mode()[0]
fare_median_buggy   = combined['Fare'].median()
```

### Why is this a problem?

When we compute the **median age** or **mode of Embarked** on the combined train+test dataset, the fill values are influenced by test set data. This means:

1. **Training imputation is contaminated** — the value used to fill missing `Age` in the training set was computed using test set passengers' ages. The model indirectly "sees" test data during training.

2. **It violates the train/test boundary** — in a real deployment scenario, you would not have access to the test (or future) data when fitting your preprocessing. Computing statistics on the full dataset pretends you do.

3. **It produces optimistic performance estimates** — because the imputed values are slightly closer to the true test distribution, the model appears to generalize better than it actually does.

The principle is: **anything fitted during preprocessing must be fitted on training data only**, and then *applied* (not re-fitted) to test data.

---
## Part 4: FIXED Pipeline (No Data Leakage)

The fix is to compute all fill statistics **only from the training set**, then apply those same statistics to the test set without re-computing them.

In [ ]:
# ---- FIXED PIPELINE ----

train_fixed = train.copy()
test_fixed  = test.copy()

# Step 1: Compute fill statistics from TRAINING DATA ONLY
age_median_fixed    = train_fixed['Age'].median()
embarked_mode_fixed = train_fixed['Embarked'].mode()[0]
fare_median_fixed   = train_fixed['Fare'].median()

print(f"[FIXED]  Age median computed on training data only: {age_median_fixed}")
print(f"[FIXED]  Embarked mode computed on training data only: {embarked_mode_fixed}")

# Step 2: Apply those training-derived statistics to BOTH train and test
train_fixed['Age']      = train_fixed['Age'].fillna(age_median_fixed)
train_fixed['Embarked'] = train_fixed['Embarked'].fillna(embarked_mode_fixed)
test_fixed['Age']       = test_fixed['Age'].fillna(age_median_fixed)      # same value, not recomputed
test_fixed['Embarked']  = test_fixed['Embarked'].fillna(embarked_mode_fixed)
test_fixed['Fare']      = test_fixed['Fare'].fillna(fare_median_fixed)

# Step 3: Feature engineering (same as before)
for df in [train_fixed, test_fixed]:
    df['Sex']      = (df['Sex'] == 'male').astype(int)
    df['Embarked'] = df['Embarked'].map({'S': 0, 'C': 1, 'Q': 2}).fillna(0)

features = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']

X_train_fixed = train_fixed[features]
y_train_fixed = train_fixed['Survived']
X_test_fixed  = test_fixed[features]

# Step 4: Train and evaluate
model_fixed = LogisticRegression(max_iter=200)
cv_scores_fixed = cross_val_score(model_fixed, X_train_fixed, y_train_fixed, cv=5)
print(f"\n[FIXED]  Cross-val accuracy: {cv_scores_fixed.mean():.4f} (+/- {cv_scores_fixed.std():.4f})")

---
## Part 5: Best Practice — Use sklearn SimpleImputer

The cleanest and most robust way to fix this is to use `sklearn`'s `SimpleImputer`, which enforces the fit-on-train / transform-both pattern explicitly.

In [ ]:
# ---- BEST PRACTICE: sklearn SimpleImputer ----

train_sk = train.copy()
test_sk  = test.copy()

# Encode sex first
for df in [train_sk, test_sk]:
    df['Sex']      = (df['Sex'] == 'male').astype(int)
    df['Embarked'] = df['Embarked'].map({'S': 0, 'C': 1, 'Q': 2})

features = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']

X_train_sk = train_sk[features].copy()
y_train_sk = train_sk['Survived']
X_test_sk  = test_sk[features].copy()

# Imputer is FIT on training data only, then TRANSFORMS both
imputer = SimpleImputer(strategy='median')
imputer.fit(X_train_sk)                            # learns medians from train only
X_train_sk = pd.DataFrame(imputer.transform(X_train_sk), columns=features)
X_test_sk  = pd.DataFrame(imputer.transform(X_test_sk),  columns=features)  # applies train medians

model_sk = LogisticRegression(max_iter=200)
cv_scores_sk = cross_val_score(model_sk, X_train_sk, y_train_sk, cv=5)
print(f"[sklearn] Cross-val accuracy: {cv_scores_sk.mean():.4f} (+/- {cv_scores_sk.std():.4f})")

---
## Part 6: Summary Comparison

In [ ]:
print("=" * 55)
print(f"{'Pipeline':<30} {'CV Accuracy':>10} {'Std':>10}")
print("=" * 55)
print(f"{'Buggy (data leakage)':<30} {cv_scores_buggy.mean():>10.4f} {cv_scores_buggy.std():>10.4f}")
print(f"{'Fixed (train stats only)':<30} {cv_scores_fixed.mean():>10.4f} {cv_scores_fixed.std():>10.4f}")
print(f"{'Best practice (sklearn)':<30} {cv_scores_sk.mean():>10.4f} {cv_scores_sk.std():>10.4f}")
print("=" * 55)
print()
print("Note: The buggy pipeline may show slightly higher accuracy,")
print("which is misleading — it is an artifact of the leakage.")

---
## Conclusion

| | Buggy Pipeline | Fixed Pipeline |
|---|---|---|
| **Fill value source** | Combined train + test | Training data only |
| **Leakage present?** | Yes | No |
| **Accuracy estimate** | Optimistically inflated | Realistic |
| **Production-safe?** | No | Yes |

**Rule to remember:** Any statistic (median, mean, mode, min, max) used in preprocessing must be **computed on training data only**, then **applied** to the test set. The test set should never influence what values are learned during preprocessing.